# Cross-asset trend — results (read-only notebook)

**规则**：notebook 只读 `reports/results/` 与 `reports/exhibits/`，不重算任何结论数字。
所有数字来自 `python run.py --config config/base.yml`；改了参数要重跑 pipeline，而不是在 notebook 里改。
这样 notebook 永远不会和 pipeline 的数字打架。

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RES, EX = ROOT / 'reports' / 'results', ROOT / 'reports' / 'exhibits'
print(json.dumps(json.loads((ROOT / 'reports' / 'manifest.json').read_text(encoding='utf-8')),
                 ensure_ascii=False, indent=2)[:900])

## 1. 主结果（净口径，含成本）

多空版本跑不赢 60/40；只做多版本 Sharpe 更高、回撤更浅。核心结论见 `reports/research_note.md`。

In [ ]:
summary = pd.read_csv(RES / 'summary.csv')
cols = ['name', 'years', 'cagr_pct', 'vol_pct', 'sharpe', 'max_dd_pct',
        'turnover_two_way_pa', 'avg_gross_exposure']
summary[cols]

## 2. 净值与回撤

红线是多空净收益，绿色虚线是只做多净收益，蓝线是 60/40。

In [ ]:
for name in ('ex2-nav', 'ex8-drawdown'):
    display(Image(filename=str(EX / f'{name}.png')))

## 3. 条件收益：它是一份保险，不是发动机

标普最差 10% 的交易日里，趋势的日均跌幅只有 60/40 的八分之一；最好的日子里它也几乎不参与。

In [ ]:
regime = pd.read_csv(RES / 'regime.csv')
display(regime[regime['kind'] == 'conditional'])
display(Image(filename=str(EX / 'ex7-stress.png')))

## 4. 稳健性与容量

30 组参数全部为正（Sharpe 0.22–0.62），说明不是单点拟合；
但 ETF 层的容量只有 50 万美元，约束来自日元 ETF 的流动性——真正实施必须走期货。

In [ ]:
grid = pd.read_csv(RES / 'param_grid.csv')
display(grid.pivot(index='lookback_days', columns='target_vol_ann', values='net_sharpe').round(2))
display(Image(filename=str(EX / 'ex6-heatmap.png')))
print(json.dumps(json.loads((RES / 'capacity.json').read_text(encoding='utf-8')),
                 ensure_ascii=False, indent=2))